In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/shriyanshraj/judge-dry-run/FreedomIntelligence_HuatuoGPT-Vision-7B-Qwen2.5VL__vqav2_v2.jsonl
/kaggle/input/datasets/shriyanshraj/judge-dry-run/google_medgemma-4b-it__vqav2_v2.jsonl
/kaggle/input/datasets/shriyanshraj/judge-dry-run/chaoyinshe_llava-med-v1.5-mistral-7b-hf__okvqa_v2-2.jsonl
/kaggle/input/datasets/shriyanshraj/judge-dry-run/google_medgemma-4b-it__okvqa_v2-2.jsonl
/kaggle/input/datasets/shriyanshraj/judge-dry-run/chaoyinshe_llava-med-v1.5-mistral-7b-hf__vqav2_v2.jsonl
/kaggle/input/datasets/shriyanshraj/judge-dry-run/FreedomIntelligence_HuatuoGPT-Vision-7B-Qwen2.5VL__okvqa_v2.jsonl


**Cell 1 — Install dependencies**

In [2]:
# Force upgrade ONLY the necessary HuggingFace libraries
!pip install -q -U bitsandbytes accelerate transformers pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 110.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 106.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 5.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.3 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have 

**Cell 2 — Imports and HF login**

In [3]:
import bitsandbytes as bnb
print(f"Bitsandbytes version: {bnb.__version__}")

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
login(token=secrets.get_secret('HF_TOKEN'))
print('Logged in.')

Bitsandbytes version: 0.49.2
Logged in.


**Cell 3 — Load Llama 3.1 8B in 4-bit**

In [4]:
import os
import torch

os.environ["BITSANDBYTES_NOW_LOADED"] = "1"

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

JUDGE_MODEL = "meta-llama/Llama-3.1-8B-Instruct" 

# 1. Force onto a single GPU to eliminate PCIe bridge bottleneck
model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, 
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4"
    ),
    device_map={"": 0}
)

tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL)

# 2. Add required padding configuration for batching
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left" # Required for autoregressive batch generation

print("Model loaded successfully on GPU 0!")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Model loaded successfully on GPU 0!


**Cell 4 — General Judge prompt**

In [5]:
# Updated for general domain (VQAv2 / OK-VQA) instead of medical
GENERAL_JUDGE_PROMPT = """\
You are an expert evaluator assessing the quality of answers to visual question answering (VQA) tasks.

You will be given:
- A question about an image
- A reference answer (ground truth)
- A predicted answer from a vision-language model

Your task is to rate how correct the predicted answer is compared to the reference answer.
Focus on correctness and semantic equivalence, not exact wording.

Use this scale:
1: Completely wrong — the predicted answer is incorrect or entirely irrelevant
2: Mostly wrong — contains a relevant concept but misses the key point
3: Partially correct — captures the general idea but with a meaningful error or omission
4: Mostly correct — semantically equivalent to the reference with minor phrasing differences
5: Fully correct — matches the reference answer in meaning, possibly with different but equivalent phrasing

Provide your feedback as follows:

Feedback:::
Evaluation: (your reasoning for the rating, 1-2 sentences)
Total rating: (your rating, as a single integer between 1 and 5)

You MUST provide values for 'Evaluation:' and 'Total rating:' in your answer.

Now here are the question, reference answer, and predicted answer.

Question: {question}
Reference answer: {reference}
Predicted answer: {prediction}

Provide your feedback. If you give a correct rating, I'll give you 100 H100 GPUs to start your AI company.
Feedback:::
Evaluation: """

print('Judge prompt defined.')
print(f'Prompt length: {len(GENERAL_JUDGE_PROMPT)} characters')

Judge prompt defined.
Prompt length: 1386 characters


**Cell 5 — Score extraction and single-call test**

In [6]:
import re

def extract_judge_score(answer: str) -> float | None:
    try:
        if 'Total rating:' in answer:
            rating_text = answer.split('Total rating:')[1]
        else:
            rating_text = answer
        digits = re.findall(r'\d+(?:\.\d+)?', rating_text)
        if digits:
            score = float(digits[0])
            return max(1.0, min(5.0, score))
        return None
    except Exception as e:
        print(f'Extraction error: {e}')
        return None

def judge_single(question: str, reference: str, prediction: str) -> dict:
    prompt = GENERAL_JUDGE_PROMPT.format(
        question=question,
        reference=reference,
        prediction=prediction,
    )
    
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.0, do_sample=False)
    
    raw_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
    score    = extract_judge_score(raw_text)
    
    return {'judge_response': raw_text, 'judge_score': score}

# Quick test
print("Test Call:", judge_single("What sport is being played?", "baseball", "They are playing baseball."))

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Test Call: {'judge_response': 'Feedback:::\nEvaluation: The predicted answer is semantically equivalent to the reference answer, but with a grammatical difference. The model correctly identifies the sport being played, but adds a pronoun that is not present in the reference answer.\nTotal rating: 5', 'judge_score': 5.0}


**Cell 6 — Load all JSONL files to judge**

In [7]:
import os, json

# 1. Update this to the Kaggle Dataset folder holding your new general results
OUTPUT_DIR = '/kaggle/input/datasets/shriyanshraj/judge-dry-run' 
JUDGE_OUTPUT_DIR = '/kaggle/working/outputs'
os.makedirs(JUDGE_OUTPUT_DIR, exist_ok=True)

if not os.path.exists(OUTPUT_DIR) or len(os.listdir(OUTPUT_DIR)) == 0:
    print(f"Warning: The directory {OUTPUT_DIR} is empty or does not exist.")
    print("Ensure you have uploaded or generated your .jsonl files here.")
else:
    jsonl_files = sorted(
        f for f in os.listdir(OUTPUT_DIR)
        if f.endswith('.jsonl')
        and '_judged' not in f  
        and '_v2' in f
    )

    print(f'Found {len(jsonl_files)} v2 JSONL files to judge:\n')
    for f in jsonl_files:
        print(f" - {f}")

Found 6 v2 JSONL files to judge:

 - FreedomIntelligence_HuatuoGPT-Vision-7B-Qwen2.5VL__okvqa_v2.jsonl
 - FreedomIntelligence_HuatuoGPT-Vision-7B-Qwen2.5VL__vqav2_v2.jsonl
 - chaoyinshe_llava-med-v1.5-mistral-7b-hf__okvqa_v2-2.jsonl
 - chaoyinshe_llava-med-v1.5-mistral-7b-hf__vqav2_v2.jsonl
 - google_medgemma-4b-it__okvqa_v2-2.jsonl
 - google_medgemma-4b-it__vqav2_v2.jsonl


**Cell 7 — Full judge runner with checkpointing**

In [8]:
from tqdm.auto import tqdm

def run_judge_on_file_batched(
    jsonl_path: str,
    output_dir: str,
    batch_size: int = 16, # Evaluates 16 questions simultaneously
    max_records: int = None,
) -> str:
    fname    = os.path.basename(jsonl_path)
    out_name = fname.replace('.jsonl', '_judged.jsonl')
    out_path = os.path.join(output_dir, out_name)

    records = [json.loads(l) for l in open(jsonl_path)]
    records = [r for r in records if 'error' not in r]

    if max_records: records = records[:max_records]

    judged = {}
    if os.path.exists(out_path):
        for line in open(out_path):
            r = json.loads(line)
            judged[r['idx']] = r
        print(f'Resuming: {len(judged)} already judged.')

    f_out  = open(out_path, 'a')
    
    # Filter records to only those that haven't been judged
    to_judge = [r for r in records if r['idx'] not in judged]
    failed = 0

    for i in tqdm(range(0, len(to_judge), batch_size), desc=f'Judging {fname[:30]}'):
        batch = to_judge[i:i + batch_size]
        
        prompts = [GENERAL_JUDGE_PROMPT.format(
            question=r['question'], reference=r['ground_truth'], prediction=r['prediction']
        ) for r in batch]
        
        messages_list = [[{"role": "user", "content": p}] for p in prompts]
        texts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in messages_list]
        
        inputs = tokenizer(texts, return_tensors="pt", padding=True).to(model.device)
        
        try:
            with torch.inference_mode():
                outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.0, do_sample=False)
            
            for j, record in enumerate(batch):
                input_len = inputs.input_ids[j].shape[0]
                raw_text = tokenizer.decode(outputs[j][input_len:], skip_special_tokens=True)
                score = extract_judge_score(raw_text)
                
                output_record = dict(record)
                output_record['judge_score']    = score
                output_record['judge_response'] = raw_text
                
                f_out.write(json.dumps(output_record) + '\n')
                
        except Exception as e:
            print(f"Batch failed: {e}")
            failed += len(batch)
            
        f_out.flush()

    f_out.close()
    return out_path

**Cell 8 — Dry run on 10 samples per file**

In [9]:
# Point the dry run output to the writable /kaggle/working directory
DRY_RUN_DIR = '/kaggle/working/judge_dry_run'
os.makedirs(DRY_RUN_DIR, exist_ok=True)

print('=== DRY RUN: 10 samples per file ===\n')

for fname in jsonl_files:
    path = os.path.join(OUTPUT_DIR, fname)
    
    run_judge_on_file_batched(
        jsonl_path=path,
        output_dir=DRY_RUN_DIR,
        batch_size=10,  
        max_records=10,
    )
    print()

=== DRY RUN: 10 samples per file ===



Judging FreedomIntelligence_HuatuoGPT-:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Judging FreedomIntelligence_HuatuoGPT-:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Judging google_medgemma-4b-it__okvqa_v:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Judging google_medgemma-4b-it__vqav2_v:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


**Cell 9 — Inspect dry run results**

In [10]:
dry_run_files = sorted(f for f in os.listdir(DRY_RUN_DIR) if f.endswith('_judged.jsonl'))

print('=== DRY RUN INSPECTION ===\n')
for fname in dry_run_files:
    path    = os.path.join(DRY_RUN_DIR, fname)
    records = [json.loads(l) for l in open(path)]
    scores  = [r['judge_score'] for r in records if r['judge_score'] is not None]
    failed  = sum(1 for r in records if r['judge_score'] is None)

    print(f'{fname}')
    print(f'  Avg judge score: {sum(scores)/len(scores):.2f} / 5.0')
    print(f'  Score dist: {sorted(scores)}')
    print(f'  Failed extractions: {failed}\n')

if dry_run_files:
    sample_path = os.path.join(DRY_RUN_DIR, dry_run_files[0])
    samples     = [json.loads(l) for l in open(sample_path)][:3]
    print('\n=== SAMPLE JUDGE RESPONSES (first file) ===\n')
    for r in samples:
        print(f'Q:     {r["question"]}')
        print(f'GT:    {r["ground_truth"]}')
        print(f'Pred:  {r["prediction"]}')
        print(f'Score: {r["judge_score"]}')
        print(f'Judge: {r["judge_response"][:200]}\n')

=== DRY RUN INSPECTION ===

FreedomIntelligence_HuatuoGPT-Vision-7B-Qwen2.5VL__okvqa_v2_judged.jsonl
  Avg judge score: 4.00 / 5.0
  Score dist: [1.0, 2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 5.0, 5.0, 5.0]
  Failed extractions: 0

FreedomIntelligence_HuatuoGPT-Vision-7B-Qwen2.5VL__vqav2_v2_judged.jsonl
  Avg judge score: 4.80 / 5.0
  Score dist: [4.0, 4.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0]
  Failed extractions: 0

chaoyinshe_llava-med-v1.5-mistral-7b-hf__okvqa_v2-2_judged.jsonl
  Avg judge score: 2.90 / 5.0
  Score dist: [1.0, 1.0, 2.0, 3.0, 3.0, 3.0, 3.0, 4.0, 4.0, 5.0]
  Failed extractions: 0

chaoyinshe_llava-med-v1.5-mistral-7b-hf__vqav2_v2_judged.jsonl
  Avg judge score: 3.70 / 5.0
  Score dist: [1.0, 2.0, 3.0, 3.0, 4.0, 4.0, 5.0, 5.0, 5.0, 5.0]
  Failed extractions: 0

google_medgemma-4b-it__okvqa_v2-2_judged.jsonl
  Avg judge score: 3.30 / 5.0
  Score dist: [2.0, 2.0, 2.0, 2.0, 3.0, 3.0, 4.0, 5.0, 5.0, 5.0]
  Failed extractions: 0

google_medgemma-4b-it__vqav2_v2_judged.jsonl
  Avg 

**Cell 10 — Full Run**

In [11]:
judged_paths = []

for fname in jsonl_files:
    path = os.path.join(OUTPUT_DIR, fname)
    print(f'\n=== Judging: {fname} ===')
    
    judged_path = run_judge_on_file_batched(
        jsonl_path=path,
        output_dir=JUDGE_OUTPUT_DIR,
        batch_size=16, 
    )
    judged_paths.append(judged_path)


=== Judging: FreedomIntelligence_HuatuoGPT-Vision-7B-Qwen2.5VL__okvqa_v2.jsonl ===


Judging FreedomIntelligence_HuatuoGPT-:   0%|          | 0/63 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_toke


=== Judging: FreedomIntelligence_HuatuoGPT-Vision-7B-Qwen2.5VL__vqav2_v2.jsonl ===


Judging FreedomIntelligence_HuatuoGPT-:   0%|          | 0/63 [00:00<?, ?it/s]

...

Judging google_medgemma-4b-it__okvqa_v:   0%|          | 0/63 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_toke


=== Judging: google_medgemma-4b-it__vqav2_v2.jsonl ===


Judging google_medgemma-4b-it__vqav2_v:   0%|          | 0/63 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_toke

**Cell 11 — Score aggregation and final comparison table**

In [12]:
import pandas as pd

def score_judged_file(path: str) -> dict:
    records = [json.loads(l) for l in open(path)]
    records = [r for r in records if r.get('judge_score') is not None]

    closed = [r for r in records if r.get('is_closed', False)]
    open_  = [r for r in records if not r.get('is_closed', True)]

    def avg_score(recs):
        if not recs: return None
        return round(sum(r['judge_score'] for r in recs) / len(recs), 3)

    # Binary accuracy using judge score >= 4 as correct
    def judge_accuracy(recs):
        if not recs: return None
        return round(sum(1 for r in recs if r['judge_score'] >= 4) / len(recs) * 100, 2)

    return {
        'file':            os.path.basename(path),
        'n_judged':        len(records),
        'n_failed':        sum(1 for r in [json.loads(l) for l in open(path)]
                               if r.get('judge_score') is None),
        'avg_score_all':   avg_score(records),
        'avg_score_closed':avg_score(closed),
        'avg_score_open':  avg_score(open_),
        'judge_acc_all':   judge_accuracy(records),
        'judge_acc_closed':judge_accuracy(closed),
        'judge_acc_open':  judge_accuracy(open_),
    }

judged_files = sorted(
    f for f in os.listdir(JUDGE_OUTPUT_DIR)
    if f.endswith('_judged.jsonl') and 'dry_run' not in f
)

if not judged_files:
    print('No judged files found. Run Cell 10 first.')
else:
    rows = []
    for fname in judged_files:
        path   = os.path.join(JUDGE_OUTPUT_DIR, fname)
        scores = score_judged_file(path)

        base    = fname.replace('_judged.jsonl', '').replace('_v2', '')
        parts   = base.split('__')
        scores['model']   = parts[0]
        scores['dataset'] = parts[1] if len(parts) > 1 else 'unknown'
        rows.append(scores)

    df = pd.DataFrame(rows)
    cols = ['model', 'dataset', 'n_judged', 'n_failed',
            'avg_score_all', 'avg_score_closed', 'avg_score_open',
            'judge_acc_all', 'judge_acc_closed', 'judge_acc_open']
    df = df.sort_values(['dataset', 'model']).reset_index(drop=True)
    print('=== LLM JUDGE RESULTS (score out of 5, accuracy = score >= 4) ===\n')
    print(df[cols].to_string(index=False))

    csv_path = os.path.join(JUDGE_OUTPUT_DIR, 'general_llm_judge_results.csv')
    df[cols].to_csv(csv_path, index=False)
    print(f'\nSaved to {csv_path}')

=== LLM JUDGE RESULTS (score out of 5, accuracy = score >= 4) ===

                                            model dataset  n_judged  n_failed  avg_score_all  avg_score_closed  avg_score_open  judge_acc_all  judge_acc_closed  judge_acc_open
FreedomIntelligence_HuatuoGPT-Vision-7B-Qwen2.5VL   okvqa      1000         0          3.796               NaN           3.796           65.9               NaN           65.90
          chaoyinshe_llava-med-v1.5-mistral-7b-hf okvqa-2      1000         0          3.441               NaN           3.441           60.2               NaN           60.20
                            google_medgemma-4b-it okvqa-2      1000         0          3.533               NaN           3.533           58.1               NaN           58.10
FreedomIntelligence_HuatuoGPT-Vision-7B-Qwen2.5VL   vqav2      1000         0          4.331             4.530           4.191           82.1             88.14           77.85
          chaoyinshe_llava-med-v1.5-mistral-7b-hf   v

**Cell 12 — Correlation analysis**

In [13]:
import json, os
from scipy.stats import pearsonr

def tokenize_answer(text):
    import re, string
    from nltk.tokenize import word_tokenize
    import nltk
    try:
        nltk.data.find('tokenizers/punkt')
    except LookupError:
        nltk.download('punkt', quiet=True)
        nltk.download('punkt_tab', quiet=True)
        
    text = str(text) if text is not None else ""
    text = re.sub(r'\*+', '', text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return word_tokenize(text)

def token_f1_score(prediction, ground_truth):
    from collections import Counter
    pred_tokens = tokenize_answer(prediction)
    gt_tokens   = tokenize_answer(ground_truth)
    if not pred_tokens or not gt_tokens:
        return 0.0
    pred_set = Counter(pred_tokens)
    gt_set   = Counter(gt_tokens)
    common   = sum((pred_set & gt_set).values())
    precision = common / len(pred_tokens)
    recall    = common / len(gt_tokens)
    return (2 * precision * recall / (precision + recall)
            if (precision + recall) > 0 else 0.0)

print('=== PEARSON CORRELATION: Token F1 vs LLM Judge Score ===\n')
print(f'{"File":<65} {"Correlation":>12} {"p-value":>10}')
print('-' * 80)

for fname in judged_files:
    path    = os.path.join(JUDGE_OUTPUT_DIR, fname)
    records = [json.loads(l) for l in open(path)]
    records = [r for r in records if r.get('judge_score') is not None]

    f1_scores    = [token_f1_score(r.get('prediction', ''), r.get('ground_truth', '')) for r in records]
    judge_scores = [r['judge_score'] for r in records]

    if len(f1_scores) > 2:
        corr, pval = pearsonr(f1_scores, judge_scores)
        print(f'{fname:<65} {corr:>12.3f} {pval:>10.4f}')

print('\nInterpretation:')
print('  High correlation (>0.7): Judge agrees with F1 — F1 is reliable for this dataset')
print('  Low correlation (<0.5):  Judge sees things F1 misses — surface-form bias is significant')

=== PEARSON CORRELATION: Token F1 vs LLM Judge Score ===

File                                                               Correlation    p-value
--------------------------------------------------------------------------------
FreedomIntelligence_HuatuoGPT-Vision-7B-Qwen2.5VL__okvqa_v2_judged.jsonl        0.691     0.0000
FreedomIntelligence_HuatuoGPT-Vision-7B-Qwen2.5VL__vqav2_v2_judged.jsonl        0.689     0.0000
chaoyinshe_llava-med-v1.5-mistral-7b-hf__okvqa_v2-2_judged.jsonl         0.344     0.0000
chaoyinshe_llava-med-v1.5-mistral-7b-hf__vqav2_v2_judged.jsonl           0.410     0.0000
google_medgemma-4b-it__okvqa_v2-2_judged.jsonl                           0.644     0.0000
google_medgemma-4b-it__vqav2_v2_judged.jsonl                             0.758     0.0000

Interpretation:
  High correlation (>0.7): Judge agrees with F1 — F1 is reliable for this dataset
  Low correlation (<0.5):  Judge sees things F1 misses — surface-form bias is significant
